# NOVA 01 — MOD-01 Obstacle Detection (YOLOv8n)
**Attach datasets** (Add Data, right sidebar):
- `jhontroya/dectectra-dataset`
- `kushagrapandya/visdrone-dataset`

**Accelerator:** GPU T4 x2 or P100. Full training ~6-9h — fits one Kaggle session (12h limit). Enable *Persistence: Files* to survive restarts.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q ultralytics onnx2tf onnx

In [ ]:
# Resolve the ACTUAL mount paths — Kaggle sometimes mounts datasets
# nested as /kaggle/input/datasets/<owner>/<slug>, so search 3 levels.
import glob
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
DETECTRA = next(p for p in inputs if 'dect' in p.split('/')[-1].lower())
VISDRONE = next((p for p in inputs if 'visdrone' in p.split('/')[-1].lower()), None)
print('Detectra:', DETECTRA)
print('VisDrone:', VISDRONE)
!find {DETECTRA} -maxdepth 3 -type d | head -20

In [ ]:
# Merges both datasets (remapping VisDrone class indices into the merged
# class list) and GENERATES the training YAML with correct nc/names.
# Aborts with a clear error if 0 images are found.
!python scripts/prepare_obstacle_dataset.py \
    --detectra {DETECTRA} \
    --visdrone {VISDRONE} \
    --out /kaggle/working/datasets/obstacle_combined \
    --yaml-out /kaggle/working/obstacle_data.yaml
!head -40 /kaggle/working/obstacle_data.yaml

In [ ]:
# Full training + TFLite INT8 export (Ultralytics native export).
# STOP if the previous cell errored — do not burn GPU hours on 0 images.
!python scripts/train_obstacle.py --data /kaggle/working/obstacle_data.yaml \
    --epochs 100 --imgsz 320 --batch 32

In [ ]:
# Publish to HuggingFace: unixio/nova-obstacle-detection
import glob
candidates = glob.glob('/kaggle/working/runs/obstacle_student/weights/**/*_int8.tflite',
                       recursive=True)
print('TFLite candidates:', candidates)
assert candidates, 'No INT8 TFLite found — training/export must succeed first.'
tflite_path = candidates[0]
best_pt = '/kaggle/working/runs/obstacle_student/weights/best.pt'
!python scripts/push_to_huggingface.py --module MOD-01 \
    --pytorch {best_pt} --tflite {tflite_path} \
    --eval-json /kaggle/working/evaluation/obstacle_results.json --version 1.0.0